# Day H5 — 종단·생존 예측: 언제 발병하는가

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/H-clinical-prediction/day-H5.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/H-clinical-prediction/day-H5.md)

In [ ]:
# Colab 기본 런타임에 없는 패키지 설치
%pip install -q lifelines

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 중도절단을 무시하면

In [ ]:
import numpy as np
import pandas as pd
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.utils import concordance_index

rng = np.random.default_rng(7)
n = 600                                         # 가상의 전구기 코호트(기준 시점에 모두 미발병)
age = rng.normal(66, 7, n)
marker = rng.normal(0, 1, n)                    # 기준 시점 지표(가상)
hazard = 0.06 * np.exp(0.05 * (age - 66) + 0.6 * marker)    # 연간 발병 위험
t_event = rng.exponential(1 / hazard)                         # 실제 발병까지 걸릴 시간(관찰 못 할 수도 있음)
t_censor = rng.uniform(1, 8, n)                               # 추적이 끝나는 시점(연구 종료·탈락)
time = np.minimum(t_event, t_censor)
event = (t_event <= t_censor).astype(int)
df = pd.DataFrame({"time": time.round(2), "event": event, "age": age.round(1), "marker": marker.round(2)})
print(df.head(4).to_string(index=False))
print(f"\n발병 관찰 {event.sum()}명 / {n}명, 중도절단 {n - event.sum()}명, 추적 중앙값 {np.median(time):.1f}년")

# 예시 1 — 중도절단을 무시하면 위험을 과소 추정한다
naive = event[time <= 5].sum() / n
km = KaplanMeierFitter().fit(df.time, df.event)
true5 = np.mean(1 - np.exp(-hazard * 5))           # 가상 세계라서 알 수 있는 "정답"
print(f"단순 비율 {naive:.3f} | KM 5년 누적 발병 {1 - km.predict(5.0):.3f} | 정답 {true5:.3f}")

### 예시 2 — KM 곡선 비교, Cox 모델, C-index

In [ ]:
# 예시 2 — Kaplan–Meier 곡선 비교, Cox 모델, 시험 세트의 C-index
df["marker_high"] = (df.marker > df.marker.median()).astype(int)
for g, sub in df.groupby("marker_high"):
    k = KaplanMeierFitter().fit(sub.time, sub.event)
    print(f"지표 {'상위' if g else '하위'} 50%: 5년 누적 발병 {1 - k.predict(5.0):.3f} (n={len(sub)}, 사건 {sub.event.sum()})")

train = rng.random(n) < 0.7
cph = CoxPHFitter().fit(df.loc[train, ["time", "event", "age", "marker"]], "time", "event")
print(cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]].round(3))
risk = cph.predict_partial_hazard(df.loc[~train, ["age", "marker"]])
# concordance_index는 "값이 클수록 오래 산다"를 가정하므로 위험 점수에 마이너스를 붙인다
print(f"시험 세트 C-index {concordance_index(df.time[~train], -risk, df.event[~train]):.3f}")

### 예시 3 — 예측 시점 이후의 값이 섞이면

In [ ]:
# 예시 3 — 예측 시점 이후의 정보가 섞이면
# 마지막 방문 때 잰 지표: 발병한 사람은 발병 무렵(진단 방문) 값이 기록되어 높게 나온다
df["marker_last_visit"] = df.marker + 1.5 * df.event + rng.normal(0, 0.5, n)
cols = ["age", "marker_last_visit"]
cph_leak = CoxPHFitter().fit(df.loc[train, ["time", "event"] + cols], "time", "event")
risk_leak = cph_leak.predict_partial_hazard(df.loc[~train, cols])
print(f"누수 변수 사용 C-index {concordance_index(df.time[~train], -risk_leak, df.event[~train]):.3f}")

## 직접 해보기

1. `notebooks/advanced/H/H5_survival.ipynb`에서 예시 1–3을 실행합니다.
2. KM 곡선을 그립니다. 예시 2 다음 셀에 붙여 넣습니다.

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

out = Path("../../../figures/advanced/H"); out.mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(figsize=(6, 4))
for g, sub in df.groupby("marker_high"):
    KaplanMeierFitter(label="marker high" if g else "marker low").fit(sub.time, sub.event).plot_survival_function(ax=ax)
ax.set_xlabel("years from baseline"); ax.set_ylabel("proportion still unconverted")
fig.tight_layout(); fig.savefig(out / "H5_km.png", dpi=150)